# 0.2 Xatu Calldata Pull

This notebook is the canonical calldata source for the bandwidth pipeline. It pulls total raw calldata bytes from Xatu's canonical beacon execution-payload transaction table, then validates zero/nonzero byte counts from Xatu's execution transaction tables.

Output:

```text
calldata_bytes = sum(canonical_beacon_block_execution_transaction.call_data_size)
calldata_gas_7999 = sum(4 * n_input_zero_bytes + 16 * n_input_nonzero_bytes)
```

## Why Xatu for Calldata

Xatu has full payload transaction coverage for calldata, and it is much cheaper to query at scale than RPC. Raw bytes come from the beacon payload transaction table. Zero/nonzero byte counts come from `execution_transaction` only after validating that it matches the beacon payload transaction count and raw byte total. `canonical_execution_transaction` is diagnostic only. RPC remains the source for BAL bytes because exact BAL reads need `prestateTracer`.

In [ ]:
import os
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.xatu_calldata import query_xatu_calldata_by_block

load_dotenv(PROJECT_ROOT / ".env")
missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing .env values: " + ", ".join(missing))

client = clickhouse_connect.get_client(
    host="clickhouse-raw.xatu.ethpandaops.io",
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)
print(client.query("SELECT version()").result_rows)

In [ ]:
NETWORK = "mainnet"
START_BLOCK = 22_886_891
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
WRITE_CSV = True

In [ ]:
calldata = query_xatu_calldata_by_block(client, BLOCKS, network=NETWORK)

calldata["execution_tx_row_delta"] = calldata["execution_tx_rows"] - calldata["n_txs_from_payload"]
calldata["execution_calldata_delta"] = calldata["execution_calldata_bytes"] - calldata["calldata_bytes"]

display(calldata)

summary = pd.DataFrame([{
    "blocks_checked": len(calldata),
    "execution_matches": int(calldata["execution_matches_beacon"].sum()),
    "canonical_execution_matches": int(calldata["canonical_execution_matches_beacon"].sum()),
    "total_payload_txs": int(calldata["n_txs_from_payload"].sum()),
    "total_calldata_bytes": int(calldata["calldata_bytes"].sum()),
    "total_zero_bytes": int(calldata["calldata_zero_bytes"].dropna().sum()),
    "total_nonzero_bytes": int(calldata["calldata_nonzero_bytes"].dropna().sum()),
    "total_calldata_gas_7999": int(calldata["calldata_gas_7999"].dropna().sum()),
    "mean_calldata_bytes_per_block": calldata["calldata_bytes"].mean(),
    "mean_calldata_gas_per_block": calldata["calldata_gas_7999"].dropna().mean(),
}])
display(summary)

if WRITE_CSV:
    data_dir = PROJECT_ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    out = data_dir / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
    calldata.to_csv(out, index=False)
    print(out)